# AIGC 5005 · Module 01 · Part 5
## Elementary Python

Companion notebook to Part 5 of the Module 01 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 45 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module01-tool` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module01-tool python=3.11 ipykernel`, then `conda activate module01-tool`. If VS Code offers to install `ipykernel`, accept.

## 0. Simulated input

Run this first. It lets the `input()` examples run without stopping to wait for you.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated input
# ---------------------------------------------------------------------------
# Notebooks cannot answer input() prompts during "Run All": execution simply
# waits for you. So that every cell runs top to bottom without stopping, this
# helper replays a list of pretend answers instead.
#
# To type the answers yourself, set USE_REAL_INPUT = True and re-run this cell.
# In a normal .py script you would just call input() directly.

USE_REAL_INPUT = False


def make_asker(answers):
    """Return an ask() function that replays `answers` in order."""
    replay = iter(answers)

    def ask(prompt):
        if USE_REAL_INPUT:
            return input(prompt)
        answer = next(replay)
        print(f"{prompt}{answer}")      # echo it, so the output reads like a real session
        return answer

    return ask


print("Simulated input ready. USE_REAL_INPUT =", USE_REAL_INPUT)

## 1. Variables are names bound to objects

A variable is a **name** attached to an object, and the attachment can change type at any time. No declaration is needed, and none is possible.

In [ ]:
count = 12          # int
print(type(count))

count = 12.5        # now a float; entirely legal
print(type(count))

count = "twelve"    # now a string; also legal
print(type(count))  # <class 'str'>

### Watch out: nothing stops a variable becoming a type you did not intend

This flexibility is convenient and is also a common source of bugs. (The `try` and `except` keeps the notebook running; you meet them properly in Module 04.) From Module 04 you will add type hints to make your intent explicit.

In [ ]:
quantity = "5"            # arrived as text, perhaps from input()

try:
    print(quantity + 1)
except TypeError as error:
    print("TypeError ->", error)

print(int(quantity) + 1)  # convert first

## 2. Numbers: division

Python has two numeric types you will use constantly. Integers have unlimited size. Floats follow the same binary representation used by every mainstream language.

In [ ]:
print("7 / 2    =", 7 / 2)       # 3.5   true division, always a float
print("7 // 2   =", 7 // 2)      # 3     floor division
print("-7 // 2  =", -7 // 2)     # -4    floors toward negative infinity, not toward zero
print("7 % 2    =", 7 % 2)       # 1     modulo
print("2 ** 10  =", 2 ** 10)     # 1024  exponentiation
print("2 ** 200 =", 2 ** 200)    # integers never overflow

**Predict first:** what is `-7 % 2`? Floor division and modulo always agree with each other: `(a // b) * b + (a % b)` gives back `a`.

In [ ]:
print("-7 % 2 =", -7 % 2)
print("check   :", (-7 // 2) * 2 + (-7 % 2) == -7)
print("divmod(-7, 2) =", divmod(-7, 2))          # both results at once

## 3. Floats and precision

Binary floating point cannot represent 0.1 exactly, so small errors appear and accumulate. This is not a Python bug.

In [ ]:
print(0.1 + 0.2)                          # 0.30000000000000004
print(0.1 + 0.2 == 0.3)                   # False: never compare floats with ==
print(abs((0.1 + 0.2) - 0.3) < 1e-9)      # True: compare within a tolerance

import math
print(math.isclose(0.1 + 0.2, 0.3))       # True: the standard library does the tolerance for you

When exactness matters, as it does for money, use the `decimal` module, which stores decimal digits exactly.

In [ ]:
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2"))                      # 0.3
print(Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))    # True
# Build Decimals from strings: Decimal(0.1) would inherit the float's error.

## 4. Rounding and conversion

Python's `round` uses **banker's rounding**: halfway cases go to the nearest **even** number. It exists to avoid upward bias when rounding large amounts of data.

**Predict first:** `round(0.5)`, `round(1.5)`, `round(2.5)`, `round(3.5)`.

In [ ]:
for value in (0.5, 1.5, 2.5, 3.5):
    print(f"round({value}) = {round(value)}")

print("round(2.675, 2) =", round(2.675, 2))   # 2.67, but not for a rounding-rule reason:
print(f"2.675 is really stored as {2.675:.20f}")   # float precision again

`int()` **truncates**, it does not round, and it only accepts strings that look like whole numbers. `float()` is more forgiving.

In [ ]:
print("int(3.9)       =", int(3.9))      # 3
print("int(-3.9)      =", int(-3.9))     # -3: truncates toward zero
print("int('7')       =", int("7"))
print("float('7.0')   =", float("7.0"))

try:
    int("7.0")                           # a whole-number VALUE, but not whole-number SYNTAX
except ValueError as error:
    print("int('7.0') -> ValueError:", error)

## 5. Reading input

`input()` always returns a **string**, whatever the user types. Convert before doing arithmetic. The weak point is the conversion: bad text stops the program with a traceback unless you catch it.

In [ ]:
ask = make_asker(["21.5"])
raw = ask("Enter a temperature in Celsius: ")
print(type(raw))                  # <class 'str'> always

celsius = float(raw)
print(celsius * 9 / 5 + 32)       # 70.7

### What does `float()` accept?

Some entries you would reject at a glance are accepted by `float()`, and some you would accept are rejected. The `try` and `except` is the pattern the worked example uses: attempt the conversion, and if it fails, say what was wrong.

**Predict first:** which of these entries convert?

In [ ]:
for raw in ["21.5", "hello", "", "  18  ", "1e3", "1,5", "nan", "inf", "-0"]:
    try:
        value = float(raw)
        print(f"{raw!r:>10} -> accepted as {value}")
    except ValueError:
        print(f"{raw!r:>10} -> ValueError")

Two surprises: whitespace is tolerated, and `nan` and `inf` are accepted as numbers. That is one reason the worked example also checks the **range** after converting. A comparison with `nan` is always false, so `MIN_C <= nan <= MAX_C` rejects it.

In [ ]:
nan = float("nan")
print(-90.0 <= nan <= 60.0)         # False: a range check catches it

## 6. Comparison, Boolean logic and truthiness

Comparisons return `True` or `False`, and Python allows **chained** comparisons, which read like mathematics.

In [ ]:
age = 30
print(18 <= age < 65)               # chained; equivalent to 18 <= age and age < 65
print(18 <= age and age < 65)

# `and` / `or` short-circuit: the right side is not evaluated if unnecessary.
count, total = 0, 100
if count > 0 and total / count > 5:  # safe: when count is 0 the division never happens
    print("average is high")
else:
    print("no division by zero happened")

Certain values count as **false** in a Boolean context: empty strings, zero, empty collections and `None`. Everything else is truthy. That is why you write `if not name:` rather than comparing with an empty string.

**Predict first:** which of these six are falsy?

In [ ]:
for value in ["", "0", 0, [], [0], None]:
    print(f"{value!r:>5} -> bool: {bool(value)}")

The string `"0"` and the list `[0]` are truthy: they are not empty.

## 7. Decisions

Python uses **indentation** to define blocks. Four spaces per level is the standard. Conditions are tested in order and the first true branch wins, so ordering from lowest to highest avoids repeating the lower bound in each test.

In [ ]:
def status(temperature):
    if temperature < 0:
        return "freezing"
    elif temperature < 18:
        return "cold"
    elif temperature < 26:
        return "comfortable"
    else:
        return "hot"


for t in (-5, 0, 10, 18, 25, 30):
    print(f"{t:>3} degrees is {status(t)}")

### Watch out: the order is part of the logic

Here the conditions are in the wrong order. **Predict first:** what does 10 degrees become?

In [ ]:
def status_wrong_order(temperature):
    if temperature < 26:
        return "comfortable"            # catches everything below 26, including 10 and -5
    elif temperature < 18:
        return "cold"                   # can never be reached
    elif temperature < 0:
        return "freezing"               # can never be reached
    else:
        return "hot"


for t in (-5, 10, 25, 30):
    print(f"{t:>3} degrees is {status_wrong_order(t)}")

## 8. f-strings

An f-string is the standard way to build text in modern Python. A **format specifier** after a colon controls how a value is displayed, and display is where formatting belongs, rather than rounding the stored value.

In [ ]:
celsius = 21.456
print(f"{celsius:.1f} C")                 # one decimal place: 21.5 C
print(f"{1234567:,}")                     # thousands separators: 1,234,567
print(f"{0.256:.1%}")                     # a percentage: 25.6%
print(f"[{'left':<8}] [{'right':>8}] [{'mid':^8}]")   # width and alignment

---
## Try it: grade boundaries

Write `letter(mark)` using `if` / `elif` / `else`: 80 or above is `"A"`, 70 to 79 is `"B"`, 60 to 69 is `"C"`, below 60 is `"F"`. Order the conditions so you never repeat a lower bound, then test the **boundary values** (59, 60, 69, 70, 79, 80), which is where decision code usually breaks.

In [ ]:
def letter(mark):
    # your code here
    pass


# for mark in (59, 60, 69, 70, 79, 80):
#     print(mark, letter(mark))